In [0]:
#Imports
from pyspark.sql.functions import *

In [0]:
#Load Bronze Tables
telemetry = spark.table("workspace.predictive_maintenance.bronze_telemetry")

errors = spark.table("workspace.predictive_maintenance.bronze_errors")

failures = spark.table("workspace.predictive_maintenance.bronze_failures")

machines = spark.table("workspace.predictive_maintenance.bronze_machines")

maint = spark.table("workspace.predictive_maintenance.bronze_maint")

In [0]:
#Check Record Counts
print("Telemetry:", telemetry.count())
print("Errors:", errors.count())
print("Failures:", failures.count())
print("Machines:", machines.count())
print("Maintenance:", maint.count())

Telemetry: 876100
Errors: 3919
Failures: 761
Machines: 100
Maintenance: 3286


In [0]:
#Clean Telemetry
telemetry_silver = (
    telemetry
    .dropDuplicates()
    .withColumn("datetime", to_timestamp("datetime"))
)

In [0]:
#check
telemetry_silver.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)



In [0]:
#Null Check Telemetry
telemetry_silver.select(
    [sum(col(c).isNull().cast("int")).alias(c)
     for c in telemetry_silver.columns]
).show()

+--------+---------+----+------+--------+---------+
|datetime|machineID|volt|rotate|pressure|vibration|
+--------+---------+----+------+--------+---------+
|       0|        0|   0|     0|       0|        0|
+--------+---------+----+------+--------+---------+



In [0]:
#Save Silver Telemetry
telemetry_silver.write \
.mode("overwrite") \
.saveAsTable("workspace.predictive_maintenance.silver_telemetry")

In [0]:
#Clean Errors
errors_silver = (
    errors
    .dropDuplicates()
    .withColumn("datetime", to_timestamp("datetime"))
)

In [0]:
#save
errors_silver.write \
.mode("overwrite") \
.saveAsTable("workspace.predictive_maintenance.silver_errors")

In [0]:
#Clean Failures
failures_silver = (
    failures
    .dropDuplicates()
    .withColumn("datetime", to_timestamp("datetime"))
)

In [0]:
#Create Failure Flag
failures_silver = failures_silver.withColumn(
    "failure_flag",
    lit(1)
)

In [0]:
#Save
failures_silver.write \
.mode("overwrite") \
.saveAsTable("workspace.predictive_maintenance.silver_failures")

In [0]:
#Clean Machines
machines_silver = (
    machines
    .dropDuplicates()
)

In [0]:
#Create Age Group
machines_silver = machines_silver.withColumn(
    "age_group",
    when(col("age") <= 5, "0-5 Years")
    .when(col("age") <= 10, "6-10 Years")
    .when(col("age") <= 15, "11-15 Years")
    .otherwise("16-20 Years")
)

In [0]:
#save
machines_silver.write \
.mode("overwrite") \
.saveAsTable("workspace.predictive_maintenance.silver_machines")

In [0]:
#Clean Maintenance
maint_silver = (
    maint
    .dropDuplicates()
    .withColumn("datetime", to_timestamp("datetime"))
)

In [0]:
#Maintenance Flag
maint_silver = maint_silver.withColumn(
    "maintenance_flag",
    lit(1)
)

In [0]:
#save
maint_silver.write \
.mode("overwrite") \
.saveAsTable("workspace.predictive_maintenance.silver_maint")

In [0]:
#Verify Silver Tables
spark.sql("""
SHOW TABLES IN workspace.predictive_maintenance
""").show(truncate=False)

+----------------------+----------------+-----------+
|database              |tableName       |isTemporary|
+----------------------+----------------+-----------+
|predictive_maintenance|bronze_errors   |false      |
|predictive_maintenance|bronze_failures |false      |
|predictive_maintenance|bronze_machines |false      |
|predictive_maintenance|bronze_maint    |false      |
|predictive_maintenance|bronze_telemetry|false      |
|predictive_maintenance|silver_errors   |false      |
|predictive_maintenance|silver_failures |false      |
|predictive_maintenance|silver_machines |false      |
|predictive_maintenance|silver_maint    |false      |
|predictive_maintenance|silver_telemetry|false      |
+----------------------+----------------+-----------+

